In [1]:
import cv2
import numpy as np
from collections import Counter

# Finds the dominant color in an image using K-Means clustering

In [2]:
def get_dominant_color(image, k=4):
    pixels = image.reshape((-1, 3))
    pixels = np.float32(pixels)

    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
    _, labels, centers = cv2.kmeans(pixels, k, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)

    centers = np.uint8(centers)
    label_counts = Counter(labels.flatten())
    dominant_cluster = label_counts.most_common(1)[0][0]

    dominant_color = centers[dominant_cluster]
    return dominant_color

# Compares the given RGB values to a predefined list and returns the closest color name

In [3]:
def get_color_name(r, g, b):
    # Dictionary of standard colors and their RGB values
    colors = {
        "Red": (255, 0, 0),
        "Green": (0, 255, 0),
        "Blue": (0, 0, 255),
        "Yellow": (255, 255, 0),
        "Orange": (255, 128, 0),
        "Purple": (128, 0, 128),
        "Pink": (255, 192, 203),
        "Cyan": (0, 255, 255),
        "Magenta": (255, 0, 255),
        "Brown": (165, 42, 42),
        "White": (255, 255, 255),
        "Black": (0, 0, 0),
        "Gray": (128, 128, 128)
    }

    min_distance = float('inf')
    closest_name = "Unknown"

    # Calculate the mathematical distance to find the closest color match
    for name, (cr, cg, cb) in colors.items():
        # We calculate the Euclidean distance squared between the colors
        distance = (r - cr)**2 + (g - cg)**2 + (b - cb)**2
        if distance < min_distance:
            min_distance = distance
            closest_name = name

    return closest_name

# Color Detector Function

In [4]:
def color_detector():
    cap = cv2.VideoCapture(0)
    
    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return

    square_size = 150 

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Error: Failed to grab frame.")
            break

        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape

        x1 = int(w / 2 - square_size / 2)
        y1 = int(h / 2 - square_size / 2)
        x2 = int(w / 2 + square_size / 2)
        y2 = int(h / 2 + square_size / 2)

        # Draw the targeting square
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 255, 255), 2)

        roi = frame[y1:y2, x1:x2]

        # Get dominant color BGR values
        dominant_bgr = get_dominant_color(roi, k=3)
        b, g, r = int(dominant_bgr[0]), int(dominant_bgr[1]), int(dominant_bgr[2])

        # Find the closest color name
        color_name = get_color_name(r, g, b)

        # --- Visual Feedback ---
        
        # Draw the color swatch box below the square
        color_swatch_start = (x1, y2 + 10)
        color_swatch_end = (x2, y2 + 65)
        cv2.rectangle(frame, color_swatch_start, color_swatch_end, (b, g, r), -1)
        cv2.rectangle(frame, color_swatch_start, color_swatch_end, (255, 255, 255), 1) 

        # Determine text color based on background brightness
        brightness = (r * 299 + g * 587 + b * 114) / 1000
        text_color = (0, 0, 0) if brightness > 125 else (255, 255, 255)

        # Display the Color Name and the RGB values inside the swatch
        cv2.putText(frame, f"Color: {color_name}", (x1 + 10, y2 + 32), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, text_color, 1)
        cv2.putText(frame, f"RGB: ({r}, {g}, {b})", (x1 + 10, y2 + 52), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, text_color, 1)

        cv2.imshow("Precision Color Detector", frame)

        # Exit loop if backtick '`' is pressed
        if cv2.waitKey(1) & 0xFF == ord('`'):
            print("Terminating program cleanly...")
            break

    cap.release()
    cv2.destroyAllWindows()

In [5]:
if __name__ == "__main__":
    color_detector()

Terminating program cleanly...
